In [ ]:
%pip install -q dotenv llama_stack_client==0.4.2

In [ ]:
import os
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [ ]:
def execute_tool_call(response):
    """Extract function tool calls from response, execute via tool_runtime, and continue."""
    function_calls = []
    for item in response.output:
        item_type = getattr(item, "type", None)
        if item_type == "function_call":
            function_calls.append(item)

    if not function_calls:
        return None

    # Build input with function call outputs
    tool_inputs = []
    for fc in function_calls:
        import json
        args = json.loads(fc.arguments) if isinstance(fc.arguments, str) else fc.arguments
        print(f"\n🔧 Calling {fc.name}({args})")

        # Execute via llama-stack tool_runtime
        result = client.tool_runtime.invoke_tool(
            tool_name=fc.name,
            kwargs=args,
        )
        output = result.content if hasattr(result, "content") else str(result)
        print(f"📋 Result: {str(output)[:300]}")

        tool_inputs.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": str(output),
        })

    # Continue conversation with tool results
    return client.responses.create(
        model=MODEL,
        input=tool_inputs,
        instructions=INSTRUCTIONS,
        tools=TOOLS,
        stream=True,
        previous_response_id=response.id,
    )


def stream_with_tools(stream):
    """Stream responses API events, handling server-side and client-side tool calls."""
    final_response = None

    for event in stream:
        event_type = getattr(event, "type", None)

        # Text streaming
        if event_type == "response.output_text.delta":
            print(getattr(event, "delta", ""), end="", flush=True)

        # Refusal
        elif event_type == "response.refusal.delta":
            print(getattr(event, "delta", ""), end="", flush=True)

        # Web search status
        elif event_type == "response.output_item.added":
            item = getattr(event, "item", None)
            if item and getattr(item, "type", None) == "web_search_call":
                print(f"\n🔍 Web search")
        elif event_type == "response.web_search_call.searching":
            print("  searching...")
        elif event_type == "response.web_search_call.completed":
            print("  search completed")

        # Capture completed response for function call handling
        elif event_type == "response.completed":
            final_response = event.response

    print()

    # If there are function (client-side) tool calls, execute and continue
    if final_response:
        next_stream = execute_tool_call(final_response)
        if next_stream:
            stream_with_tools(next_stream)  # recurse for multi-turn

In [ ]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")
tavily_search_api_key = os.getenv("TAVILY_SEARCH_API_KEY")
wolfram_alpha_api_key = os.getenv("WOLFRAM_ALPHA_API_KEY")

client = LlamaStackClient(
    base_url=base_url,
    provider_data={"tavily_search_api_key": tavily_search_api_key, "wolfram_alpha_api_key": wolfram_alpha_api_key},
)

In [ ]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = """You are a helpful websearch assistant. When you are asked to search the latest you must use a tool. 
Whenever a tool is called, be sure return the response in a friendly and helpful tone."""

TOOLS = [
    {"type": "web_search"},
    {
        "type": "function",
        "name": "wolfram_alpha",
        "description": "Query WolframAlpha for computational knowledge and math calculations",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The query to compute"}
            },
            "required": ["query"],
        },
    },
]

In [6]:
question = "Who is the current F1 World Champion?"

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
    max_infer_iters=3,
)

stream_with_tools(stream)

In [8]:
question = "(7 * 12 ^ 10) / 321 ? and How many calories are there in a pound of strawberries?"

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
    max_infer_iters=3,
)

stream_with_tools(stream)